In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/candor_dataset_clean.csv')

# Linear Mixed Model
Unilateral model controls for:
* Some people are more positive in general
* Some calls are more positive in general

Mutual model assumes:

* Each call is one independent unit

# Linear Mixed Model : one sided

A Mixed Model =  Fixed effects (normal regression effects) + Random effects (group-level variation)

* Fixed effects: On average, across all conversations and speakers, how do partner behaviors (TFO, overlap, pauses, speech activity) affect actor enjoyment?
* Random effects: 
    - Each conversation (dyad) gets its own intercept because some conversations are naturally more enjoyable than others. Model estimates baseline_enjoyment + u_call (how much this convo differs from avg)
    - Each speaker also gets their own intercept. baseline_enjoyment + u_call + u_speaker (how much this speaker differs from avg)

Allows correct p-values and realistic modeling of dyadic data.

Some convos naturally fun and some ppl naturally fun. If no control - we may think partner tfo predicts enjoyment when it was just a fun convo

In [4]:
import statsmodels.formula.api as smf

model = smf.mixedlm(
    "how_enjoyable_actor ~ tfo_partner + overlap_partner + pauses_partner + speech_activity_partner",
    data=df,
    groups=df["call_id"],          
    vc_formula={"Speaker": "0 + C(actor_id)"})
result = model.fit()
print(result.summary())

c:\Users\amb20\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


              Mixed Linear Model Regression Results
Model:             MixedLM Dependent Variable: how_enjoyable_actor
No. Observations:  3200    Method:             REML               
No. Groups:        1600    Scale:              1.1552             
Min. group size:   2       Log-Likelihood:     -5881.8047         
Max. group size:   2       Converged:          Yes                
Mean group size:   2.0                                            
------------------------------------------------------------------
                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------
Intercept                8.054    0.111 72.721 0.000  7.837  8.271
tfo_partner             -0.912    0.147 -6.218 0.000 -1.199 -0.624
overlap_partner         -0.307    0.768 -0.400 0.689 -1.812  1.198
pauses_partner          -0.662    0.271 -2.444 0.015 -1.193 -0.131
speech_activity_partner -0.575    0.155 -3.724 0.000 -0.878 -0.273
Speaker Va

* When partner TFO increases by 1 unit -> actor enjoyment decreases ~ 0.91 units
* No evidence that partner overlap predicts actor enjoyment.
* More pauses by the partner are associated with lower actor enjoyment.
* Higher partner speech activity -> lower actor enjoyment.

##  R**2
How much variance is explained by fixed effects only? (TFO, Overlap, Pauses, Speech Activity)

## Condiitonalr R**2
How much variance is explained by fixed + random effects together? (partner behaviours and speaker differences)

In [5]:
print(result.cov_re)
print(result.vcomp)

Empty DataFrame
Columns: []
Index: []
[1.15515004]


In [6]:
var_residual = result.scale
var_random = result.vcomp[0]

# predictions 
fixed_preds = result.model.exog @ result.fe_params

# Variance of fixed effects
var_fixed = np.var(fixed_preds, ddof=1)

var_total = var_fixed + var_random + var_residual

r2_marginal = var_fixed / var_total
r2_conditional = (var_fixed + var_random) / var_total

print("Marginal R²:", r2_marginal)
print("Conditional R²:", r2_conditional)

Marginal R²: 0.020121336460907708
Conditional R²: 0.5100606682304539


* Only 2% of the variance in actor enjoyment is explained by partner behaviors (TFO, overlap, pauses, speech activity).
* 51% of the variance in actor enjoyment is explained when we include speaker differences

Predictors are statistically significant but effect size is small

## Linear Mixed Model : dyad

Do the combined conversational behaviors of both participants predict how mutually enjoyable the call was?

In [ ]:
# Aggregate to call level
df_mutual = df.groupby("call_id").agg({
    "how_enjoyable_partner": "mean", 
    "tfo_actor": "mean",
    "tfo_partner": "mean",
    "overlap_actor": "mean",
    "overlap_partner": "mean",
    "pauses_actor": "mean",
    "pauses_partner": "mean",
    "speech_activity_actor": "mean",
    "speech_activity_partner": "mean"
}).reset_index()
df_mutual

,call_id,how_enjoyable_partner,tfo_actor,tfo_partner,overlap_actor,overlap_partner,pauses_actor,pauses_partner,speech_activity_actor,speech_activity_partner
0,0020a0c5-1658-4747-99c1-2839e736b481,8.5,0.4355,0.4355,0.0525,0.0525,0.1610,0.1610,0.3975,0.3975
1,002d68da-7738-4177-89d9-d72ae803e0e4,8.0,0.3460,0.3460,0.0270,0.0270,0.1825,0.1825,0.3950,0.3950
2,00411458-8275-4b92-a000-d52187f03604,7.5,0.2015,0.2015,0.0515,0.0515,0.1340,0.1340,0.4150,0.4150
3,00ae2f18-9599-4df6-8e3a-6936c86b97f0,8.0,0.2790,0.2790,0.0405,0.0405,0.2090,0.2090,0.3675,0.3675
4,00b410f7-8b5f-4404-8433-0fb8c4be8f62,7.0,0.3335,0.3335,0.0455,0.0455,0.2840,0.2840,0.3245,0.3245
...,...,...,...,...,...,...,...,...,...,...
1595,ffcc79cc-cac5-49db-aea5-6cf893e7394c,7.5,0.3945,0.3945,0.0515,0.0515,0.2485,0.2485,0.3455,0.3455
1596,ffd372b5-0a7e-420f-bcd5-459119723e89,6.0,0.3175,0.3175,0.0335,0.0335,0.1110,0.1110,0.4060,0.4060
1597,ffe29987-3502-4f72-9987-cc5208ae3880,8.5,0.5675,0.5675,0.0240,0.0240,0.1965,0.1965,0.3660,0.3660
1598,ffe5cccf-82b3-4938-a2e9-38335d188e44,9.0,-0.0850,-0.0850,0.1465,0.1465,0.1755,0.1755,0.4350,0.4350


In [ ]:

"""model = smf.mixedlm(
    "how_enjoyable_actor ~ tfo_partner + overlap_partner + pauses_partner + speech_activity_partner",
    data=df,
    groups=df["call_id"]  )     

result = model.fit()
print(result.summary())"""